In [2]:
import pandas as pd
from itables import init_notebook_mode
import polars as pl
import os
import numpy as np
init_notebook_mode(all_interactive=True)
import polars.selectors as cs



In [3]:
import sys
sys.path.append("/home/a379i/Scripts")   # path to folder containing the python file

from utils.load_gtf_cgc_dresden import *
from ProteinExpression.load_pr_data import *

/home/a379i/Scripts/utils/load_gtf_cgc_dresden.py:106: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Extended predisp' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  cgc.loc[cgc["geneID_short"].isin(extended_dresden_dt["geneID_short"]), "Predisposition"] = "Extended predisp"


In [4]:
sa = pd.read_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/sample_data/master_drop_sample_annotation_sizeFactorFiltered_0.1.tsv", sep="\t")

fr_res =  pd.read_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/drop_runs/drop_master_202502_allGenes/processed_results/aberrant_splicing/results/v19/fraser/aggregated_outliers_variants.tsv", sep="\t")
# fr_res = pd.merge(fr_res, sa, left_on="sampleID", right_on="pid")




In [9]:
fr_res

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [5]:
fr_predisp = fr_res[fr_res["hgncSymbol"].isin(extended_dresden_list)]
fr_samples = fr_predisp["sampleID"].unique()
len(fr_samples)

1923

In [5]:
snvs = pd.read_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/vep_res_aggregated/vep_res_rare_snv_all_aggregated_unique_variant_type.tsv", nrows=1000, sep="\t")

In [ ]:
import polars as pl

snvs = (
    pl.scan_csv(
        "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/vep_res_aggregated/vep_res_rare_snv_all_aggregated.tsv", 
        separator="\t"
    )
    # 1. Broad filter ONLY for Gene and Sample first
    .filter(
        (pl.col("SYMBOL").is_in(extended_dresden_list)) & 
        (pl.col("sampleID").is_in(fr_samples))
    )
    # 2. Calculate score, but handle nulls so the code doesn't crash
    .with_columns(
        max_spliceai_score = pl.when(pl.col("SpliceAI_pred").is_not_null())
        .then(
            pl.col("SpliceAI_pred")
            .str.split("|")
            .list.slice(1, 4) 
            .list.eval(pl.element().cast(pl.Float64))
            .list.max()
        )
        .otherwise(pl.lit(0.0)) # Assign 0.0 if missing, so the filter can handle it
    )
    # 3. Now apply the OR logic
    .filter(
        (pl.col("max_spliceai_score") >= 0.2) | 
        (pl.col("IMPACT") == "HIGH") | 
        (pl.col("Consequence").str.contains("spl"))
    )
    .collect(engine="streaming")
)


In [ ]:
impact_map = {'HIGH': 4, 'MODERATE': 3, 'LOW': 2, 'MODIFIER': 1}
df_final = (
    snvs
    # 1. Map IMPACT to rank (using replace)
    .with_columns(
        impact_rank = pl.col("IMPACT").replace(impact_map, default=0).cast(pl.Int8)
    )
    # 2. Filter out missing Gene symbols
    .filter(pl.col("Gene") != "-")
    # 3. Sort (descending list must match columns list)
    .sort(
        by=['sampleID', 'Gene', '#Uploaded_variation', 'impact_rank'], 
        descending=[False, False, False, True]
    )
    # 4. Drop duplicates (keeping the first occurrence, which is the highest rank)
    .unique(
        subset=['sampleID', 'Gene', '#Uploaded_variation'], 
        keep="first"
    )
)

df_final.write_csv(
    "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/vep_res_aggregated/predisp/predisp_splicing_snvs.tsv", 
    separator="\t"
)

/tmp/ipykernel_2077146/6579649.py:5: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  impact_rank = pl.col("IMPACT").replace(impact_map, default=0).cast(pl.Int8)


In [ ]:
import polars as pl

indels = (
    pl.scan_csv(
        "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/vep_res_aggregated/vep_res_rare_indel_all_aggregated.tsv", 
        separator="\t"
    )
    # 1. Broad filter ONLY for Gene and Sample first
    .filter(
        (pl.col("SYMBOL").is_in(extended_dresden_list)) & 
        (pl.col("sampleID").is_in(fr_samples))
    )
    # 2. Calculate score, but handle nulls so the code doesn't crash
    .with_columns(
        max_spliceai_score = pl.when(pl.col("SpliceAI_pred").is_not_null())
        .then(
            pl.col("SpliceAI_pred")
            .str.split("|")
            .list.slice(1, 4) 
            .list.eval(pl.element().cast(pl.Float64))
            .list.max()
        )
        .otherwise(pl.lit(0.0)) # Assign 0.0 if missing, so the filter can handle it
    )
    # 3. Now apply the OR logic
    .filter(
        (pl.col("max_spliceai_score") >= 0.2) | 
        (pl.col("IMPACT") == "HIGH") | 
        (pl.col("Consequence").str.contains("spl"))
    )
    .collect(engine="streaming")
)

df_final_indels = (
    indels
    # 1. Map IMPACT to rank (using replace)
    .with_columns(
        impact_rank = pl.col("IMPACT").replace(impact_map, default=0).cast(pl.Int8)
    )
    # 2. Filter out missing Gene symbols
    .filter(pl.col("Gene") != "-")
    # 3. Sort (descending list must match columns list)
    .sort(
        by=['sampleID', 'Gene', '#Uploaded_variation', 'impact_rank'], 
        descending=[False, False, False, True]
    )
    # 4. Drop duplicates (keeping the first occurrence, which is the highest rank)
    .unique(
        subset=['sampleID', 'Gene', '#Uploaded_variation'], 
        keep="first"
    )
)

df_final_indels = df_final_indels.drop("")
df_final_indels.write_csv(
    "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/vep_res_aggregated/predisp/predisp_splicing_indels.tsv", 
    separator="\t"
)


/tmp/ipykernel_2077146/3573921207.py:38: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  impact_rank = pl.col("IMPACT").replace(impact_map, default=0).cast(pl.Int8)


## check how manny high intron varinats

In [6]:
df_final = pl.read_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/vep_res_aggregated/predisp/predisp_splicing_snvs.tsv", separator="\t")

df_final_indels = pl.read_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/vep_res_aggregated/predisp/predisp_splicing_indels.tsv", separator="\t")


df_final = df_final.with_columns(type = pl.lit("snv"))
df_final_indels = df_final_indels.with_columns(type = pl.lit("indel"))

# Now use the "relaxed" concat to handle the SchemaError you had earlier
all_vars = pl.concat([df_final, df_final_indels], how="diagonal_relaxed")


In [73]:
all_vars.write_csv(
    "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/vep_res_aggregated/predisp/predisp_splicing_all_vars.tsv", 
    separator="\t"
)

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [7]:
fr_res_vars = fr_predisp.merge(all_vars.to_pandas(), left_on=["sampleID", "hgncSymbol"], right_on=["sampleID", "SYMBOL"])

In [8]:
duplicates = fr_res_vars.duplicated(subset=["sampleID", "geneID"])
fr_res_vars[duplicates]

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [13]:
fr_predisp[fr_predisp["IMPACT"].notna()]
fr_predisp[(fr_predisp["IMPACT"].notna()) & 
           ((fr_predisp["ANNOTATION_control_snv"].str.contains("germline", na=False)) |
            (fr_predisp["ANNOTATION_control_indel"].str.contains("germline", na=False)))]

c = fr_res_vars[(fr_res_vars["IMPACT_x"].notna()) & 
           ((fr_res_vars["ANNOTATION_control_snv"].str.contains("germline", na=False)) |
            (fr_res_vars["ANNOTATION_control_indel"].str.contains("germline", na=False)))]

c[c["IMPACT_y"] != "HIGH"] # non-canonical


c[(c["IMPACT_y"] != "HIGH")]

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [19]:
# c = pd.merge(c, sa, left_on="sampleID", right_on="pid")

c[(c["IMPACT_y"] != "HIGH") & (c["Consequence"] == "intron_variant")][[ "AF", "Consequence", "SYMBOL", "IMPACT_y", "pid", "start", "end", "deltaPsi",  "max_spliceai_score", "potentialImpact", "Oncotree Code"]]

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [15]:
c

Loading ITables v2.5.2 from the init_notebook_mode cell... (need help?)


In [18]:
c.columns[50:]

Index(['alt_indel', '#Uploaded_variation', 'Location', 'Allele', 'Gene_y',
       'IMPACT_y', 'Consequence', 'SYMBOL', 'PHENO', 'STRAND_x',
       'Feature_type', 'Feature', 'am_class', 'am_pathogenicity', 'LoF',
       'LoF_filter', 'AF', 'LoF_flags', 'LoF_info', 'SpliceAI_pred',
       'CADD_PHRED', 'CADD_RAW', 'max_spliceai_score', 'impact_rank', 'type_y',
       'gnomADe_AF', 'gnomADg_AF', 'Unnamed: 0', 'pid', 'Oncotree Code', 'SEX',
       'Birthdate', 'Deathdate', 'Date of Diagnosis', 'ICD10 Code', 'freitext',
       'Oncotree Text', 'Diagnosis valid', 'Proteome data', 'ILSE IDs',
       'MTB Date', 'ICD10_prefix', 'TISSUE', 'DROP_GROUP', 'sample_type_base',
       'sample_type', 'PAIRED_END', 'STRAND_y', 'ratio', 'RNA_BAM_FILE',
       'seq_type', 'DNA_VCF_FILE', 'RNA_ID', 'DNA_ID', 'COUNT_MODE',
       'COUNT_OVERLAPS', 'ANNOTATION', 'GENE_COUNTS_FILE', 'oncotree_num',
       'ICD10_num', 'tissue_num', 'indel_vcf', 'snv_vcf', 'sv_vcf', 'cnv_vcf',
       'bam_file', 'Diag_y', 'p